# Extreme Precipitation Event Template

Config-driven analysis of an extreme precipitation event. Everything specific to the
event (station, dates, event definitions, warming levels, map area) comes from
`event_config.yaml` in this folder (`<post>/_analysis/`).

**Events:** each entry in `thresholds.events` is an `accumulation_days`-day rainfall total
above a threshold (a `percentile` or a fixed `threshold_in`, see Part 3). Back-to-back
qualifying days count as one event.

## Setup

In [ ]:
import sys

# Run from blog/<post>/_analysis/ — the shared helpers are three folders up.
sys.path.insert(0, "../../../relevant-hazard-template/helpers")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import event_helpers as eh

cfg = eh.load_config()
if cfg.hazard != "precip":
    raise ValueError(f"This is the precip notebook but the config says hazard: {cfg.hazard}")
cfg.make_output_dirs()

WET_DAY_IN = cfg.thresholds.get("wet_day_in", 0.01)
EVENTS = cfg.thresholds["events"]
for ev in EVENTS:
    missing = {"id", "label", "accumulation_days"} - set(ev)
    if missing:
        raise KeyError(f"thresholds.events entry {ev} is missing {sorted(missing)}")
    if ("percentile" in ev) == ("threshold_in" in ev):
        raise KeyError(f"thresholds.events entry {ev['id']} needs exactly one of percentile / threshold_in")

# Cal-CRAI climate metrics combined with the event metric in Part 5. The CRAI's own
# projected-precip metrics (precip_99percentile, surface_runoff) are left out — the
# event metric replaces them.
EXPOSURE_METRICS = cfg.crai_exposure_metrics or ["median_flood_warning_days", "floodplain_percentage"]
LOSS_METRICS = cfg.crai_loss_metrics or [
    "avg_flood_insurance_payout_per_claim",
    "estimated_flood_crop_loss_cost",
    "total_flood_fatalities",
]

_first = pd.Timestamp(cfg.event_windows[0][0])
EVENT_WATER_YEAR = _first.year + (_first.month >= 10)   # Oct–Dec belong to the next water year
print(f"{len(EVENTS)} event definition(s): {[ev['id'] for ev in EVENTS]}; event water year {EVENT_WATER_YEAR}")

---

## Part 1: What Happened — Observed Station Data

Hourly precipitation from HDP, summed to daily totals in
inches (UTC days). The station is `location.station_label` (an airport code) in
`location.network_id`; if the label is empty, the station in that network inside
`location.clip` with the longest precipitation record is used.

> **Role in the post:** grounds the opening story in data, feeding "How Was This Event
> Defined?", the "event by the numbers" table, and the closer look at the storm.

In [ ]:
if cfg.station_label:
    station_id = eh.resolve_hdp_station(cfg.station_label, cfg.network_id)
else:
    # No airport code: the station with the longest pr record inside the clip county
    station_id = eh.find_station_in_area(eh.county_boundary(cfg.clip), cfg.network_id)
    cfg.station_label = eh.station_icao(station_id) or station_id   # used in figure titles
station = eh.pull_hdp_station(station_id, cfg.network_id)

daily = eh.hdp_daily_precip_in(station)
n_years = (daily.time.values[-1] - daily.time.values[0]) / np.timedelta64(1, "D") / 365.25
print(f"Record: {str(daily.time.values[0])[:10]} to {str(daily.time.values[-1])[:10]} "
      f"({n_years:.0f} years, {int(daily.notnull().sum())} days with reports)")

### The event, by the numbers

Totals in each configured window (`event.windows`) are listed. Afterwards, the storms are defined, the station threshold for extreme precip listed, how many such events are on record, and where this one ranks.

In [ ]:
def window_label(i, start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    e_str = e.strftime("%-d") if s.month == e.month else e.strftime("%b %-d")
    return f"Storm {i} ({s.strftime('%b %-d')}–{e_str})"


for i, (start, end) in enumerate(cfg.event_windows, start=1):
    d = daily.sel(time=slice(start, end))
    print(window_label(i, start, end))
    print(f"  Total        : {float(d.sum()):.2f} in")
    print(f"  Max 1-day    : {float(d.max()):.2f} in")
    for ev in EVENTS:
        n = ev["accumulation_days"]
        peak = float(eh.nday_totals(daily, n, "time").sel(time=slice(start, end)).max())
        print(f"  Max {n}-day    : {peak:.2f} in")
    print()

station_thresholds = {}
for ev in EVENTS:
    n = ev["accumulation_days"]
    if "threshold_in" in ev:   # fixed amount
        thr = float(ev["threshold_in"])
        thr_note = "fixed amount"
    else:                      # percentile of the station record
        thr = float(eh.wet_day_nday_threshold(daily, n, ev["percentile"], WET_DAY_IN, "time"))
        thr_note = f"{ev['percentile']:.0%} of {n}-day totals ending on a wet day"
    station_thresholds[ev["id"]] = thr

    totals = eh.nday_totals(daily, n, "time").to_series()
    exceed = totals > thr
    starts = exceed & ~exceed.shift(1, fill_value=False)
    run_id = starts.cumsum()
    peaks = totals[exceed].groupby(run_id[exceed]).max().sort_values(ascending=False)

    print(f"{ev['label']} ({ev['id']})")
    print(f"  Station threshold : {thr:.2f} in over {n} days ({thr_note})")
    print(f"  Events on record  : {len(peaks)} ({len(peaks) / n_years:.2f}/yr, about 1-in-{n_years / max(len(peaks), 1):.1f} years)")
    for i, (start, end) in enumerate(cfg.event_windows, start=1):
        peak = totals[start:end].max()
        if peak > thr:
            rank = int((peaks > peak).sum()) + 1
            print(f"  Storm {i}           : qualifies — {peak:.2f} in, #{rank} of {len(peaks)} on record")
        else:
            print(f"  Storm {i}           : does NOT qualify ({peak:.2f} in ≤ {thr:.2f} in)")
    print()

### Cumulative precipitation by water year

Running total from October 1 for every water year on record, with the event's water
year in bold and the event windows shaded.

In [ ]:
df = daily.to_pandas().rename("precip_in").reset_index()
df["wy"] = eh.water_year(df["time"])
df["cum_in"] = df["precip_in"].fillna(0).groupby(df["wy"]).cumsum()
# Shared Oct-1 → Sep-30 axis (anchored on water year 2000, a leap year, so every year lines up)
df["plot_date"] = pd.to_datetime({"year": np.where(df["time"].dt.month >= 10, 1999, 2000),
                                  "month": df["time"].dt.month, "day": df["time"].dt.day})


def anchor_wy2000(date_str):
    d = pd.Timestamp(date_str)
    return f"{1999 if d.month >= 10 else 2000}-{d.month:02d}-{d.day:02d}"


hist_years = sorted(set(df["wy"]) - {EVENT_WATER_YEAR})
hover_years = {y for y in hist_years if y % 5 == 0} | {hist_years[0]}

fig = go.Figure()
for wy, grp in df[df["wy"] != EVENT_WATER_YEAR].groupby("wy"):
    hoverable = wy in hover_years
    fig.add_trace(go.Scatter(
        x=grp["plot_date"], y=grp["cum_in"], mode="lines",
        line=dict(color="#9ecae1", width=0.8), opacity=0.4, showlegend=False,
        hoverinfo=None if hoverable else "skip",
        hovertemplate=f"WY {wy}<br>%{{x|%b %d}}<br>%{{y:.2f}} in<extra></extra>" if hoverable else None,
    ))
ev_df = df[df["wy"] == EVENT_WATER_YEAR]
fig.add_trace(go.Scatter(
    x=ev_df["plot_date"], y=ev_df["cum_in"], mode="lines",
    line=dict(color="#08306b", width=2.4), name=f"Water year {EVENT_WATER_YEAR}",
    hovertemplate=f"WY {EVENT_WATER_YEAR}<br>%{{x|%b %d}}<br>%{{y:.2f}} in<extra></extra>",
))
for i, (start, end) in enumerate(cfg.event_windows, start=1):
    fig.add_vrect(x0=anchor_wy2000(start), x1=anchor_wy2000(end), fillcolor="darkgoldenrod",
                  opacity=0.2, line_width=0, layer="below",
                  annotation_text=window_label(i, start, end).replace(" (", "<br>("),
                  annotation_position="top left", annotation_font=dict(size=11, color="darkgoldenrod"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#9ecae1", width=1.6),
                         name="Other water years"))

fig.update_layout(
    title=f"{cfg.place_name} ({cfg.station_label}) — Cumulative Precipitation by Water Year",
    xaxis=dict(title="", range=["1999-10-01", "2000-09-30"], showgrid=False, fixedrange=True, tickformat="%b"),
    yaxis=dict(title="Cumulative precipitation (in)", showgrid=False, fixedrange=True),
    width=815, height=600,
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
    hovermode="closest", dragmode=False,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60, r=30, t=50, b=50),
)
fig.show(config=eh.PLOTLY_CONFIG)
eh.save_figure(fig, cfg, "cumulative_precip_by_year")

---

## Part 2: Historical Context- How Unusual Was This Year?

Each water year's total relative to the record mean, with a 5-year rolling mean. Only water
years with reports on at least 90% of days are used; years after the event are left out.

> **Role in the post:** answers "how bad was this, historically?" Did the storm come in an
> unusually wet year, or was it a short, intense burst in an ordinary one?

In [ ]:
wy = eh.water_year(daily.time).rename("water_year")
wy_total = daily.groupby(wy).sum(min_count=1)
# Share of a full water year with reports — a partial first/last year counts as incomplete.
wy_coverage = daily.notnull().groupby(wy).sum() / 365
keep = (wy_coverage >= 0.9) & (wy_total["water_year"] <= EVENT_WATER_YEAR)
wy_total = wy_total.where(keep, drop=True)

anomaly = wy_total - float(wy_total.mean())
print(f"Record mean: {float(wy_total.mean()):.1f} in/yr over {wy_total.size} complete water years")

fig = eh.plot_annual_bars(
    anomaly["water_year"].values, anomaly.values, EVENT_WATER_YEAR,
    title=f"Water-Year Precipitation Anomaly relative to record mean at {cfg.station_label}",
    yaxis_title="Precipitation anomaly (in)", unit="in",
    positive_color="steelblue", negative_color="peru",
)
eh.save_figure(fig, cfg, "annual_precip_anomaly")

---

## Part 3: How Frequently Will This Occur in the Future?

Dynamically downscaled **WRF** daily precipitation (UCLA, 3 km, SSP3-7.0) for
`location.clip` at each warming level. For each event definition:

1. **Threshold** — either a fixed amount (`threshold_in`, the same everywhere), or a
   `percentile` of `accumulation_days`-day totals ending on a wet day, calibrated per
   simulation and grid cell at `projection.baseline_warming_level`. Calibrating removes
   each model's wet/dry bias; a fixed amount keeps it but shows where that amount is
   reached most often.
2. **Event counting** — distinct events above the threshold at each warming level.
3. **Aggregation** — events per year, averaged across grid cells then simulations, as a
   return period (1-in-X years).

> **Role in the post:** the headline finding and usually a "Key Message" (e.g. "about 3x
> more likely by mid-century"). It turns the story from what happened to what's coming,
> and sets up the impacts: more frequent events leave less time to recover.

In [ ]:
# Pulled once and reused by Part 4 — simulations in projection.exclude_simulations are dropped.
# float32 halves the memory of the rolling sums below.
prec = eh.pull_wrf_warming_levels("prec", cfg, "inches/d").astype("float32")
print(dict(prec.sizes))
print("simulations:", list(prec.sim.values))

In [ ]:
gridded = {}   # per-cell events per year, for Part 4
for ev in EVENTS:
    n = ev["accumulation_days"]
    if "threshold_in" in ev:   # same fixed amount everywhere, no calibration
        threshold = float(ev["threshold_in"])
    else:                      # calibrated per simulation + cell at the baseline -> (sim, y, x)
        threshold = eh.wet_day_nday_threshold(
            prec.sel(warming_level=cfg.baseline_warming_level), n, ev["percentile"], WET_DAY_IN, "time_delta"
        )
    events = eh.nday_events(prec, n, threshold, "time_delta")   # (sim, warming_level, y, x)
    events_per_year = events.mean(["x", "y"]).mean("sim") / (2 * cfg.warming_level_window)

    fig, return_periods = eh.plot_return_periods(
        events_per_year.values, cfg, title=f"How Much More Often Will {ev['label']} Happen?"
    )
    eh.save_figure(fig, cfg, f"return_period_{ev['id']}")
    print(ev["label"])
    for wl, rate, rp in zip(cfg.warming_levels, events_per_year.values, return_periods):
        print(f"  {wl}°C: {rate:.3f} events/yr -> 1-in-{rp:.1f} yrs")

    gridded[f"{ev['id']}_events"] = events.mean("sim") / (2 * cfg.warming_level_window)

---

## Part 4: Where — Tract-Level Event Frequency

Part 3's per-cell events per year (averaged across simulations) mapped onto census tracts:
each tract takes its nearest WRF grid cell. Values are written to
`crai_layers/tract_metrics/` and min-max standardized within the area for Part 5. The maps
here and in Part 5 draw only tracts whose center is inside the zip-code area.

With a `percentile` threshold the baseline map is near-uniform by construction, so **the
change across warming levels is the signal**; `threshold_in` maps vary across the area.

> **Role in the post:** shifts from "how often" to "where," so readers can find their own
> neighborhood, and leads into who is most vulnerable.

In [ ]:
zip_codes = eh.load_zip_codes(cfg.zip_shapefile)
tracts = eh.tracts_in_area(cfg, zip_codes)
zip_labels = eh.geoid_to_zip_label(tracts, zip_codes)

tract_layers = {metric: eh.tract_metric_layers(da, metric, tracts, cfg) for metric, da in gridded.items()}

In [ ]:
for ev in EVENTS:
    metric = f"{ev['id']}_events"
    eh.tabbed_choropleth(
        tract_layers[metric], metric, f"{ev['label']} — Events per Year",
        f"{metric}_by_gwl", cfg, tracts, zip_labels, zip_codes,
        hover_label="Events per year", colorbar_title="Events / year", colorscale="Blues",
    )

---

## Part 5: Who is impacted- Climate Risk and Adaptation Index (CRAI)

For each event metric and warming level:

1. **Hazard score** — the event metric joins the Cal-CRAI flood exposure and loss metrics
   (`crai.exposure_metrics` / `crai.loss_metrics`, defaults set in Setup), combined as
   exposure × loss and standardized statewide.
2. **CRAI score** — adaptive capacity (built + governance + natural + society domain
   scores, from `crai_cache/`) divided by the hazard score.

Written to `crai_layers/hazard_scores/` and `crai_layers/crai_scores/`. The maps show the
change in hazard score at each warming level relative to the baseline.

> **Role in the post:** shows where most vulnerable communities to rising flood hazard are, and providing motivation for supporting solutions to address these issues.

In [ ]:
adaptive_capacity, climate_metrics = eh.load_crai_cache(cfg)

hazard_layers, crai_layers = {}, {}
for metric, layers in tract_layers.items():
    hazard_layers[metric], crai_layers[metric] = {}, {}
    for gwl in cfg.gwl_order:
        hazard = eh.hazard_scores(layers[gwl], metric, climate_metrics, EXPOSURE_METRICS, LOSS_METRICS)
        crai = eh.crai_scores(hazard, adaptive_capacity)
        hazard.to_csv(cfg.hazard_score_csv(metric, gwl), index=False)
        crai.to_csv(cfg.crai_score_csv(metric, gwl), index=False)
        hazard_layers[metric][gwl] = tracts.merge(hazard, on="GEOID")
        crai_layers[metric][gwl] = tracts.merge(crai, on="GEOID")
    print(metric, {g: round(float(crai_layers[metric][g]["crai_score"].median()), 3) for g in cfg.gwl_order},
          "(median CRAI score in area)")

In [ ]:
for ev in EVENTS:
    metric = f"{ev['id']}_events"
    deltas = eh.deltas_vs_baseline(hazard_layers[metric], "hazard_score", cfg.gwl_order)
    # More events lower the hazard score (CRAI flips each metric), so flip the sign:
    # positive = more precip events, drawn dark blue at the top of the colorbar
    for d in deltas.values():
        d["delta"] = -d["delta"]
    eh.tabbed_delta_choropleth(
        deltas, f"{ev['label']} — Difference from Baseline", f"{metric}_delta_by_gwl",
        cfg, zip_labels, zip_codes,
        reversescale=False, ticktext=("Fewer <br>precip events", "More <br>precip events"),
    )

---

## Part 6: So What?

TODO

> **Role in the post:** the impacts section — what the numbers mean for people and
> infrastructure here (flooding, levees and drainage, roads and transit, public health, who
> is hit hardest), gathered from reporting and published sources.

## Part 7: What's Next?

TODO

> **Role in the post:** ends on action — solutions for this hazard, what is already being
> done locally, and what readers can do (today, in their community, as policymakers).